In [ ]:
# hide
import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
def neural_lm(params: list[np.ndarray], contexts: np.ndarray) -> np.ndarray:
    """P(next note | previous C notes), computed by a small neural network: look up an
    embedding for each note of the context, concatenate them, apply a hidden layer,
    and turn the result into a probability for each pitch (a "softmax")."""
    E, W1, b1, W2, b2 = params
    h = np.tanh(E[contexts].reshape(len(contexts), -1) @ W1 + b1)
    z = h @ W2 + b2
    p = np.exp(z - z.max(axis=1, keepdims=True))
    return p / p.sum(axis=1, keepdims=True)

In [ ]:
# collapse
def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12), tuned with an all-pass filter."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)
    M = int(P - 0.6)
    D = P - 0.5 - M
    c = (1 - D) / (1 + D)
    g = 10 ** (-3 * P / (f_s * duration))
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)


def load_melody(midi_path: str) -> tuple[list[int], float, float]:
    """The melody of a MIDI file (its highest note at each onset), as in Chapter 14,
    along with its median IOI and duration, in seconds."""
    score, metronome = pq.Score.from_midi(midi_path)
    highest = {}
    for time, kwargs in score:
        if time not in highest or kwargs["pitch"] > highest[time]["pitch"]:
            highest[time] = kwargs
    onsets = sorted(highest)
    seconds = [metronome.tick_to_seconds(time) for time in onsets]
    ioi = float(np.median(np.diff(seconds)))
    duration = float(np.median([highest[time]["duration"] for time in onsets]))
    return [highest[time]["pitch"] for time in onsets], ioi, duration


def make_examples(sequence: np.ndarray, C: int) -> tuple[np.ndarray, np.ndarray]:
    """Every (context of C notes, next note) pair in a sequence."""
    contexts = np.stack([sequence[i:len(sequence) - C + i] for i in range(C)], axis=1)
    return contexts, sequence[C:]


def bits_per_note(p: np.ndarray, targets: np.ndarray) -> float:
    """The average negative log-likelihood of the targets, in bits per note."""
    return float(-np.mean(np.log2(p[np.arange(len(targets)), targets])))


def ngram(train: np.ndarray, n: int, V: int, contexts: np.ndarray, k: float = 0.1) -> np.ndarray:
    """Fit an n-gram model by counting (Chapter 14), adding k to every count so that
    unseen n-grams get a small probability, then predict the next note for each context."""
    counts = np.zeros((V,) * n)
    for i in range(len(train) - n + 1):
        counts[tuple(train[i:i + n])] += 1
    probabilities = (counts + k) / (counts + k).sum(axis=-1, keepdims=True)
    if n == 1:                                      # a unigram ignores the context
        return np.tile(probabilities, (len(contexts), 1))
    return probabilities[tuple(contexts[:, len(contexts[0]) - n + 1:].T)]


def train_neural_lm(train: np.ndarray, V: int, C: int, m: int = 8, H: int = 32,
                    steps: int = 300, lr: float = 0.01, weight_decay: float = 0.01) -> list:
    """Minimize the negative log-likelihood of the training data by gradient descent
    (with the Adam update rule), computing gradients by backpropagation."""
    rng = np.random.default_rng(0)
    params = [rng.normal(0, 0.1, (V, m)), rng.normal(0, 0.1, (C * m, H)), np.zeros(H),
              rng.normal(0, 0.1, (H, V)), np.zeros(V)]          # E, W1, b1, W2, b2
    contexts, targets = make_examples(train, C)
    m1, m2 = [np.zeros_like(p) for p in params], [np.zeros_like(p) for p in params]
    for t in range(1, steps + 1):
        x = params[0][contexts].reshape(len(contexts), -1)       # forward pass
        h = np.tanh(x @ params[1] + params[2])
        dz = neural_lm(params, contexts)                         # backward pass
        dz[np.arange(len(targets)), targets] -= 1
        dz /= len(targets)
        dh = dz @ params[3].T * (1 - h ** 2)
        dE = np.zeros_like(params[0])
        np.add.at(dE, contexts.reshape(-1), (dh @ params[1].T).reshape(-1, m))
        grads = [dE, x.T @ dh, dh.sum(axis=0), h.T @ dz, dz.sum(axis=0)]
        for i, (param, grad) in enumerate(zip(params, grads)):   # Adam update
            grad = grad + weight_decay * param
            m1[i] = 0.9 * m1[i] + 0.1 * grad
            m2[i] = 0.999 * m2[i] + 0.001 * grad ** 2
            param -= lr * (m1[i] / (1 - 0.9 ** t)) / (np.sqrt(m2[i] / (1 - 0.999 ** t)) + 1e-8)
    return params

In [ ]:
pitches, ioi, duration = load_melody("../assets/mario.mid")
states = sorted(set(pitches))
sequence = np.array([states.index(p) for p in pitches])
split = int(0.8 * len(sequence))          # fit to the first 80%, evaluate on the rest
train, test = sequence[:split], sequence[split:]
C = 4                                     # every model sees (at most) 4 previous notes
contexts, targets = make_examples(np.concatenate([train[-C:], test]), C)

for n in range(1, C + 2):
    print(f"{n}-gram:    {bits_per_note(ngram(train, n, len(states), contexts), targets):.2f} bits per note")
params = train_neural_lm(train, len(states), C)
print(f"neural LM: {bits_per_note(neural_lm(params, contexts), targets):.2f} bits per note")

In [ ]:
sequence = [states.index(p) for p in pitches[:C]]   # start from the first C notes
while len(sequence) < 64:
    p = neural_lm(params, np.array([sequence[-C:]]))[0]
    sequence.append(np.random.choice(len(states), p=p))
score = pq.Score([(i * ioi, {"pitch": states[s], "duration": duration})
                  for i, s in enumerate(sequence)])
pq.play(score.render(pluck), normalize=True)